In [15]:
from pyspark.sql.functions import current_timestamp, to_timestamp, col

raw_path = "Files/raw/*.csv"

df_raw = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .csv(raw_path)
)

df_raw = df_raw.toDF(
    "SalesOrderNumber",
    "SalesOrderLineNumber",
    "OrderDate",
    "CustomerName",
    "EmailAddress",
    "ProductName",
    "OrderQuantity",
    "UnitPrice",
    "SalesAmount"
)

# convert order date
df_raw = df_raw.withColumn(
    "OrderDate",
    to_timestamp(col("OrderDate"))
)

# ✅ ADD ingestion column
df_raw = df_raw.withColumn(
    "IngestionDate",
    current_timestamp()
)

# overwrite bronze completely
df_raw.write.mode("overwrite").saveAsTable("bronze_sales")

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 17, Finished, Available, Finished, False)

In [16]:
df_clean = df_raw.withColumn(
    "IngestionDate",
    current_timestamp()
)
print(df_clean)

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 18, Finished, Available, Finished, False)

DataFrame[SalesOrderNumber: string, SalesOrderLineNumber: int, OrderDate: timestamp, CustomerName: string, EmailAddress: string, ProductName: string, OrderQuantity: int, UnitPrice: double, SalesAmount: double, IngestionDate: timestamp]


In [17]:
control_df = spark.sql("""
SELECT last_processed_date
FROM pipeline_control
WHERE pipeline_name = 'sales_pipeline'
""")
print(control_df)
last_date = control_df.collect()[0][0]

df_clean = spark.table("bronze_sales") \
    .filter(col("IngestionDate") > last_date)
    

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 19, Finished, Available, Finished, False)

DataFrame[last_processed_date: timestamp]


In [18]:
df_clean.write.mode("overwrite").saveAsTable("silver_sales")

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 20, Finished, Available, Finished, False)

In [19]:
df_clean.createOrReplaceTempView("updates")

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 21, Finished, Available, Finished, False)

In [20]:
%%sql
MERGE INTO silver_sales AS target
USING updates AS source
ON target.SalesOrderNumber = source.SalesOrderNumber
AND target.SalesOrderLineNumber = source.SalesOrderLineNumber

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *


StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [21]:
%%sql
UPDATE pipeline_control
SET last_processed_date = current_timestamp()
WHERE pipeline_name = 'sales_pipeline'


StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [22]:
df_clean.count()

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 24, Finished, Available, Finished, False)

32718

In [23]:
from pyspark.sql.functions import *

gold_df = (
    spark.table("silver_sales")
    .groupBy(
        to_date("OrderDate").alias("OrderDate"),
        col("ProductName")
    )
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("OrderQuantity").alias("TotalQuantity"),
        sum("SalesAmount").alias("TotalRevenue")
    )
)

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 25, Finished, Available, Finished, False)

In [24]:
gold_df.write.mode("overwrite").saveAsTable("gold_sales_summary")

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 26, Finished, Available, Finished, False)

In [25]:
gold_df.createOrReplaceTempView("gold_updates")

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 27, Finished, Available, Finished, False)

In [26]:
%%sql
MERGE INTO gold_sales_summary AS target
USING gold_updates AS source
ON target.OrderDate = source.OrderDate
AND target.ProductName = source.ProductName

WHEN MATCHED THEN UPDATE SET
    target.TotalOrders = source.TotalOrders,
    target.TotalQuantity = source.TotalQuantity,
    target.TotalRevenue = source.TotalRevenue

WHEN NOT MATCHED THEN INSERT *


StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 28, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [27]:
%%sql
SELECT *
FROM gold_sales_summary
ORDER BY OrderDate DESC
LIMIT 20;


StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 29, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 5 fields>

In [30]:
# Get list of already processed files
processed = []

if spark.catalog.tableExists("processed_files"):
    processed = (
        spark.table("processed_files")
        .select("file_name")
        .rdd.flatMap(lambda x: x)
        .collect()
    )

files = mssparkutils.fs.ls("Files/raw")

file_list = [f.path for f in files]

new_files = [f for f in file_list if f not in processed]

print("New files:", new_files)

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 32, Finished, Available, Finished, False)

New files: []


In [33]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Explicit schema for headerless tab-delimited sales files
sales_schema = StructType([
    StructField("SalesOrderNumber", StringType(), True),
    StructField("SalesOrderLineNumber", IntegerType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("EmailAddress", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("OrderQuantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("SalesAmount", DoubleType(), True)
])

print("Schema loaded successfully")

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 35, Finished, Available, Finished, False)

Schema loaded successfully


In [34]:
df_raw = (
    spark.read
    .option("header","false")
    .option("delimiter","\t")
    .schema(sales_schema)
    .csv(new_files)
)

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 36, Finished, Available, Finished, False)

In [38]:
from pyspark.sql import Row
from datetime import datetime

if len(new_files) > 0:

    rows = [
        Row(file_name=f, processed_timestamp=datetime.now())
        for f in new_files
    ]

    spark.createDataFrame(rows) \
        .write.mode("append") \
        .saveAsTable("processed_files")

    print("Processed files logged.")

else:
    print("No files to log.")
 

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 40, Finished, Available, Finished, False)

No files to log.


In [39]:
%%sql
CREATE TABLE IF NOT EXISTS gold_sales (
    SalesOrderNumber STRING,
    SalesOrderLineNumber INT,
    OrderDate DATE,
    CustomerName STRING,
    EmailAddress STRING,
    ProductName STRING,
    OrderQuantity INT,
    UnitPrice DOUBLE,
    SalesAmount DOUBLE
);

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [40]:
%%sql
MERGE INTO gold_sales AS target
USING silver_sales AS source
ON target.SalesOrderNumber = source.SalesOrderNumber
AND target.SalesOrderLineNumber = source.SalesOrderLineNumber

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 42, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [42]:
bad_records = df_raw.filter(col("SalesAmount").isNull())

print("Bad records:", bad_records.count())

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 44, Finished, Available, Finished, False)

Bad records: 0


In [43]:
df = spark.sql("SELECT * FROM Sales_Trial_Project_6.dbo.gold_sales LIMIT 1000")
display(df)

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f8a9970e-38ba-4bed-b0e5-39aed44f6fbb)

In [44]:
df = spark.sql("SELECT * FROM Sales_Trial_Project_6.dbo.pipeline_control LIMIT 1000")
display(df)

StatementMeta(, 4c7644a5-b433-4de7-9d36-e933241c754a, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 51a398b4-67e9-4c74-a0c2-980761b9f681)